In [2]:
library(DBI)
library(RPostgres)
library(tidyverse)
library(glue)
library(httr)
library(jsonlite)
library(dplyr)
library(tidyr)
library(lubridate)
library(stringr)
library(ggplot2)
library(readr)

source("Pitcher_Pitch_Selection_Static_Data.r")
source("Pitch_Selection_Multipliers.r")
source("Pitcher_First_Pitch_Prediction.r")
source("Pitch_Prediction.r")
source("Scouting_Report.r")
source("Full_Pitcher_Pitch_Scouting_Report.r")
source("sql_functions.r")



statcast_data <- read_csv("C:/Users/james.villegas/OneDrive - Rotork plc/Desktop/Baseball/statcast_23_26.csv", show_col_types = FALSE)

# clean data

statcast_data_cleaned <- statcast_data %>%
drop_na(
    pitch_type
    )

# re arrange 
statcast_data_ordered <- statcast_data_cleaned %>%
arrange(
        game_date,
        game_pk,
        at_bat_number,
        pitch_number
        )

# filter out <= 1000 pitches, not qualified pitchers

statcast_data_grouped <- statcast_data_ordered %>%
group_by(
    pitcher
    ) %>%
summarise(
    count = n()
    ) %>%
filter(
    count >= 1000
    ) %>%
select(
    pitcher
    )

# order data correctly to make sure in order

batch_size <- 50

pitcher_id_list <- statcast_data_grouped$pitcher

pitcher_batches <- split(pitcher_id_list, ceiling(seq_along(pitcher_id_list) / batch_size))

batch_index_list <- seq(1, length(names(pitcher_batches)))

for (batch in batch_index_list) {
    
    message("Working on Batch : ", batch)
    
    current_batch <- pitcher_batches[[batch]]
    
    for (pitcher in current_batch) {
    
        cache_file <- paste0("cache/pitcher_", pitcher, ".rds")
    
        if (file.exists(cache_file)) {
            message("Skipping cached pitcher: ", pitcher)
            next
        }
        
        message("Running report on: ", pitcher)
        
        pitchers_report <- run_pitcher_scouting_report(pitcher, statcast_data_ordered)
    
        saveRDS(pitchers_report, cache_file)
        }  
}


all_pitcher_report_list <- list()

for (batch in batch_index_list) {

    current_batch <- pitcher_batches[[batch]]

    message("Working on batch : ", batch)

    for (pitcher in current_batch) {

        cache_file <- paste0("cache/pitcher_", pitcher, ".rds")

        if (!file.exists(cache_file)) {
            message("Does not exist, creating pitcher report for: ", pitcher)

            pitcher_report <- run_pitcher_scouting_report(pitcher, statcast_data_ordered)

            saveRDS(pitcher_report, cache_file)

        } else {
            message("Grabbing pitcher report for: ", pitcher)
            pitcher_report <- readRDS(cache_file)
        }

        # ALWAYS append the report
        all_pitcher_report_list[[length(all_pitcher_report_list) + 1]] <- pitcher_report
    }
}


all_pitcher_reports_df <- bind_rows(all_pitcher_report_list)

pitch_names <- statcast_data %>%
    distinct(
        pitch_type,
        pitch_name
        )

final_all_pitcher_reports_df <- all_pitcher_reports_df %>%
    left_join(
        pitch_names,
        by = "pitch_type"
    ) %>%
    left_join(
        pitch_names %>%
            rename(
                prev_pitch = pitch_type,
                prev_pitch_name = pitch_name
            ),
        by = "prev_pitch"
    ) %>%
    relocate(
        pitch_name, .before = probability
    ) %>%
    relocate(
        prev_pitch_name, .after = prev_pitch
    )


sql_table_name <- 'pitcher_scouting_reports'

write_df_to_sql(sql_table_name, final_all_pitcher_reports_df)

New names:
• `` -> `...1`
Working on Batch : 1

Skipping cached pitcher: 425794

Skipping cached pitcher: 425844

Skipping cached pitcher: 434378

Skipping cached pitcher: 445276

Skipping cached pitcher: 445926

Skipping cached pitcher: 446372

Skipping cached pitcher: 448179

Skipping cached pitcher: 450203

Skipping cached pitcher: 453286

Skipping cached pitcher: 455119

Skipping cached pitcher: 456501

Skipping cached pitcher: 458677

Skipping cached pitcher: 458681

Skipping cached pitcher: 471911

Skipping cached pitcher: 472610

Skipping cached pitcher: 477132

Skipping cached pitcher: 489119

Skipping cached pitcher: 489446

Skipping cached pitcher: 493603

Skipping cached pitcher: 500779

Skipping cached pitcher: 502043

Skipping cached pitcher: 502085

Skipping cached pitcher: 502171

Skipping cached pitcher: 502624

Skipping cached pitcher: 506433

Skipping cached pitcher: 518397

Skipping cached pitcher: 518489

Skipping cached pitcher: 518585

Skipping cached pitcher: 518

pitcher_id,pitcher_name,situation_id,balls,strikes,stance,runners_on,risp,tto,prev_pitch,prev_pitch_name,prev_result,pitch_type,pitch_name,probability
<dbl>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
425794,"Wainwright, Adam",1,0,0,R,FALSE,FALSE,1,NA,NA,NA,SI,Sinker,60.80
425794,"Wainwright, Adam",1,0,0,R,FALSE,FALSE,1,NA,NA,NA,CU,Curveball,15.54
425794,"Wainwright, Adam",1,0,0,R,FALSE,FALSE,1,NA,NA,NA,FC,Cutter,23.66
425794,"Wainwright, Adam",2,0,1,R,FALSE,FALSE,1,SI,Sinker,S,SI,Sinker,14.85
425794,"Wainwright, Adam",2,0,1,R,FALSE,FALSE,1,SI,Sinker,S,CU,Curveball,46.06
425794,"Wainwright, Adam",2,0,1,R,FALSE,FALSE,1,SI,Sinker,S,FC,Cutter,39.09
425794,"Wainwright, Adam",3,0,1,R,FALSE,FALSE,1,CU,Curveball,S,SI,Sinker,24.73
425794,"Wainwright, Adam",3,0,1,R,FALSE,FALSE,1,CU,Curveball,S,CU,Curveball,41.20
425794,"Wainwright, Adam",3,0,1,R,FALSE,FALSE,1,CU,Curveball,S,FC,Cutter,34.07
